# Import Libraries

## Setting Up PySpark

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, expr
import seaborn as sns
import pandas as pd
from pyspark.sql import functions as F

In [ ]:
spark = SparkSession.builder.appName("PySpark Preprocessing").getOrCreate()

## Helper Functions

In [ ]:
def check_duplicates(df):
    return df.subtract(df.distinct()).count()

In [ ]:
def n_distinct(df, col_name):
  return df.select(col_name).distinct().count()

In [ ]:
def check_missing_values(train_bureau):
    missing_values = train_bureau.select([count(when(col(c).isNull(), c)).alias(c) for c in train_bureau.columns])

    missing_values = missing_values.collect()[0].asDict()

    missing_cols = {col: count for col, count in missing_values.items() if count > 0}

    if missing_cols:
        print("Columns with missing values:")
        for col_name, missing_count in missing_cols.items():
            print(f"{col_name}: {missing_count} missing values")
    else:
        print("There are no missing values in the dataset.")

In [ ]:
def naRows_count(df, col_name):
  return df.filter(col(col_name).isNull()).count()

In [ ]:
def printInfo(df):
  print(f"Entries count: {df.count()}")
  print(f"Data columns (total {len(df.columns)} columns)")
  df.printSchema()

In [ ]:
def printShape(df):
  return (df.count(), len(df.columns))

In [ ]:
def get_numeric_col(df):
  numeric_types = ['int', 'long', 'float', 'double']
  return [col_name for col_name, dtype in df.dtypes if any(numeric_type in dtype for numeric_type in numeric_types)]

In [ ]:
def get_numeric(df):
  return df.select(*get_numeric_col(df))

In [ ]:
def get_categorical_col(df):
  return [col_name for col_name, dtype in df.dtypes if dtype == "string"]

In [ ]:
def get_categorical(df):
  return df.select(*get_categorical_col(df))

In [ ]:
def calculate_quantile(df):
  percentiles = [0.25, 0.5, 0.75]
  numeric_cols = get_numeric_col(df)
  summary_data = []
  for column in numeric_cols:
      quantiles = train_bureau_num.approxQuantile(column, percentiles, 0.01)
      summary_data.append({
          "column": column,
          "25%": quantiles[0],
          "50%": quantiles[1],
          "75%": quantiles[2]
      })
  return summary_data

## Explore Bureau Dataset

In [ ]:
HC_bureau = spark.read.csv('HC_bureau.csv', header=True, inferSchema=True)

In [ ]:
HC_bureau.limit(5).toPandas()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN


In [ ]:
printInfo(HC_bureau)

Entries count: 1716428
Data columns (total 17 columns)
root
 |-- SK_ID_CURR: integer (nullable = true)
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- CREDIT_ACTIVE: string (nullable = true)
 |-- CREDIT_CURRENCY: string (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- CREDIT_DAY_OVERDUE: integer (nullable = true)
 |-- DAYS_CREDIT_ENDDATE: double (nullable = true)
 |-- DAYS_ENDDATE_FACT: double (nullable = true)
 |-- AMT_CREDIT_MAX_OVERDUE: double (nullable = true)
 |-- CNT_CREDIT_PROLONG: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)
 |-- AMT_CREDIT_SUM_LIMIT: double (nullable = true)
 |-- AMT_CREDIT_SUM_OVERDUE: double (nullable = true)
 |-- CREDIT_TYPE: string (nullable = true)
 |-- DAYS_CREDIT_UPDATE: integer (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)



# Home Credit Default Risk

## Explore Bureau Balance Dataset and join two Bureau Datasets

In [ ]:
HC_bureau_balance = spark.read.csv('HC_bureau_balance.csv', header=True, inferSchema=True)

In [ ]:
HC_bureau_balance.limit(5).toPandas()

,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C


In [ ]:
printInfo(HC_bureau_balance)

Entries count: 27299925
Data columns (total 3 columns)
root
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- MONTHS_BALANCE: integer (nullable = true)
 |-- STATUS: string (nullable = true)



In [ ]:
# Perform INNER JOIN on the column 'SK_ID_BUREAU'
train_bureau = HC_bureau.join(HC_bureau_balance, on="SK_ID_BUREAU", how="inner")

# Display the number of rows and columns
printShape(train_bureau)

(24179741, 19)

In [ ]:
train_bureau.limit(5).toPandas()

,SK_ID_BUREAU,SK_ID_CURR,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,MONTHS_BALANCE,STATUS
0,5001712,162368,Closed,currency 1,-568,0,-264.0,-264.0,0.0,0,138388.5,0.0,0.0,0.0,Consumer credit,-261,NaN,0,C
1,5001712,162368,Closed,currency 1,-568,0,-264.0,-264.0,0.0,0,138388.5,0.0,0.0,0.0,Consumer credit,-261,NaN,-1,C
2,5001712,162368,Closed,currency 1,-568,0,-264.0,-264.0,0.0,0,138388.5,0.0,0.0,0.0,Consumer credit,-261,NaN,-2,C
3,5001712,162368,Closed,currency 1,-568,0,-264.0,-264.0,0.0,0,138388.5,0.0,0.0,0.0,Consumer credit,-261,NaN,-3,C
4,5001712,162368,Closed,currency 1,-568,0,-264.0,-264.0,0.0,0,138388.5,0.0,0.0,0.0,Consumer credit,-261,NaN,-4,C


In [ ]:
printInfo(train_bureau)

Entries count: 24179741
Data columns (total 19 columns)
root
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- CREDIT_ACTIVE: string (nullable = true)
 |-- CREDIT_CURRENCY: string (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- CREDIT_DAY_OVERDUE: integer (nullable = true)
 |-- DAYS_CREDIT_ENDDATE: double (nullable = true)
 |-- DAYS_ENDDATE_FACT: double (nullable = true)
 |-- AMT_CREDIT_MAX_OVERDUE: double (nullable = true)
 |-- CNT_CREDIT_PROLONG: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)
 |-- AMT_CREDIT_SUM_LIMIT: double (nullable = true)
 |-- AMT_CREDIT_SUM_OVERDUE: double (nullable = true)
 |-- CREDIT_TYPE: string (nullable = true)
 |-- DAYS_CREDIT_UPDATE: integer (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- MONTHS_BALANCE: integer (nullable = true)
 |-- STATUS: string (nullable = true)



Check all values in features

In [ ]:
list_item = []
for col_name in train_bureau.columns:
    print(f"Processing column {col_name}...")
    na = naRows_count(train_bureau, col_name)
    unique_sample = train_bureau.select(col_name).distinct()
    list_item.append([col_name, train_bureau.schema[col_name].dataType, na,
                      100 * na / train_bureau.count(), unique_sample.count(),
                      train_bureau.select(col_name).distinct().rdd.flatMap(lambda x: x).collect()])

desc_df = spark.createDataFrame(list_item, ['feature', 'data_type', 'null_num', 'null_percent', 'unique_num', 'unique_sample'])

Processing column SK_ID_BUREAU...
Processing column SK_ID_CURR...
Processing column CREDIT_ACTIVE...
Processing column CREDIT_CURRENCY...
Processing column DAYS_CREDIT...
Processing column CREDIT_DAY_OVERDUE...
Processing column DAYS_CREDIT_ENDDATE...
Processing column DAYS_ENDDATE_FACT...
Processing column AMT_CREDIT_MAX_OVERDUE...
Processing column CNT_CREDIT_PROLONG...
Processing column AMT_CREDIT_SUM...
Processing column AMT_CREDIT_SUM_DEBT...
Processing column AMT_CREDIT_SUM_LIMIT...
Processing column AMT_CREDIT_SUM_OVERDUE...
Processing column CREDIT_TYPE...
Processing column DAYS_CREDIT_UPDATE...
Processing column AMT_ANNUITY...
Processing column MONTHS_BALANCE...
Processing column STATUS...


In [ ]:
desc_df.show()

+--------------------+---------+--------+--------------------+----------+--------------------+
|             feature|data_type|null_num|        null_percent|unique_num|       unique_sample|
+--------------------+---------+--------+--------------------+----------+--------------------+
|        SK_ID_BUREAU|       {}|       0|                 0.0|    774354|[5001711, 5001712...|
|          SK_ID_CURR|       {}|       0|                 0.0|    134542|[293243, 176073, ...|
|       CREDIT_ACTIVE|       {}|       0|                 0.0|         4|[Bad debt, Sold, ...|
|     CREDIT_CURRENCY|       {}|       0|                 0.0|         4|[currency 2, curr...|
|         DAYS_CREDIT|       {}|       0|                 0.0|      2923|[-362, -565, -200...|
|  CREDIT_DAY_OVERDUE|       {}|       0|                 0.0|       410|[148, 1127, 243, ...|
| DAYS_CREDIT_ENDDATE|       {}| 1177501|   4.869783344660309|     12406|[305.0, 934.0, 49...|
|   DAYS_ENDDATE_FACT|       {}| 5628643|  23.2783

- Check Duplicates:

In [ ]:
check_duplicates(train_bureau)

0

There is no duplicate.

- Find Features that have Missing Values:

In [ ]:
check_missing_values(train_bureau)

Columns with missing values:
DAYS_CREDIT_ENDDATE: 1177501 missing values
DAYS_ENDDATE_FACT: 5628643 missing values
AMT_CREDIT_MAX_OVERDUE: 17545903 missing values
AMT_CREDIT_SUM: 5 missing values
AMT_CREDIT_SUM_DEBT: 4089077 missing values
AMT_CREDIT_SUM_LIMIT: 10376144 missing values
AMT_ANNUITY: 9553957 missing values


- Filter Features into Numerical and Categorical types:

In [ ]:
numeric_types = ['int', 'long', 'float', 'double']

numeric_cols = [col_name for col_name, dtype in train_bureau.dtypes if any(numeric_type in dtype for numeric_type in numeric_types)]

train_bureau_num = train_bureau.select(*numeric_cols)

train_bureau_num.show(3, truncate=False)

+------------+----------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------------+-------------------+--------------------+----------------------+------------------+-----------+--------------+
|SK_ID_BUREAU|SK_ID_CURR|DAYS_CREDIT|CREDIT_DAY_OVERDUE|DAYS_CREDIT_ENDDATE|DAYS_ENDDATE_FACT|AMT_CREDIT_MAX_OVERDUE|CNT_CREDIT_PROLONG|AMT_CREDIT_SUM|AMT_CREDIT_SUM_DEBT|AMT_CREDIT_SUM_LIMIT|AMT_CREDIT_SUM_OVERDUE|DAYS_CREDIT_UPDATE|AMT_ANNUITY|MONTHS_BALANCE|
+------------+----------+-----------+------------------+-------------------+-----------------+----------------------+------------------+--------------+-------------------+--------------------+----------------------+------------------+-----------+--------------+
|5001712     |162368    |-568       |0                 |-264.0             |-264.0           |0.0                   |0                 |138388.5      |0.0                |0.0                 |0.0                   

In [ ]:
categorical_cols = [col_name for col_name, dtype in train_bureau.dtypes if dtype == "string"]

train_bureau_cat = train_bureau.select(*categorical_cols)

train_bureau_cat.show(3, truncate=False)

+-------------+---------------+-----------+------+
|CREDIT_ACTIVE|CREDIT_CURRENCY|CREDIT_TYPE|STATUS|
+-------------+---------------+-----------+------+
|Active       |currency 1     |Credit card|X     |
|Active       |currency 1     |Credit card|0     |
|Active       |currency 1     |Credit card|0     |
+-------------+---------------+-----------+------+
only showing top 3 rows



In [ ]:
summary_data = calculate_quantile(train_bureau)

In [ ]:
summary_data

[{'column': 'SK_ID_BUREAU',
  '25%': 5731156.0,
  '50%': 6070820.0,
  '75%': 6424177.0},
 {'column': 'SK_ID_CURR', '25%': 186232.0, '50%': 277654.0, '75%': 366698.0},
 {'column': 'DAYS_CREDIT', '25%': -2282.0, '50%': -1544.0, '75%': -978.0},
 {'column': 'CREDIT_DAY_OVERDUE', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'DAYS_CREDIT_ENDDATE',
  '25%': -1674.0,
  '50%': -925.0,
  '75%': -71.0},
 {'column': 'DAYS_ENDDATE_FACT',
  '25%': -1813.0,
  '50%': -1196.0,
  '75%': -708.0},
 {'column': 'AMT_CREDIT_MAX_OVERDUE', '25%': 0.0, '50%': 0.0, '75%': 0.45},
 {'column': 'CNT_CREDIT_PROLONG', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'AMT_CREDIT_SUM',
  '25%': 48757.5,
  '50%': 114750.0,
  '75%': 276043.5},
 {'column': 'AMT_CREDIT_SUM_DEBT', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'AMT_CREDIT_SUM_LIMIT', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'AMT_CREDIT_SUM_OVERDUE', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'DAYS_CREDIT_UPDATE',
  '25%': -1216.0,
  '50%'

In [ ]:
summary_df = spark.createDataFrame(summary_data).toPandas().set_index("column")
summary_df

,25%,50%,75%
column,,,
SK_ID_BUREAU,5731156.0,6070820.0,6424177.000
SK_ID_CURR,186232.0,277654.0,366698.000
DAYS_CREDIT,-2282.0,-1544.0,-978.000
CREDIT_DAY_OVERDUE,0.0,0.0,0.000
DAYS_CREDIT_ENDDATE,-1674.0,-925.0,-71.000
DAYS_ENDDATE_FACT,-1813.0,-1196.0,-708.000
AMT_CREDIT_MAX_OVERDUE,0.0,0.0,0.450
CNT_CREDIT_PROLONG,0.0,0.0,0.000
AMT_CREDIT_SUM,48757.5,114750.0,276043.500


In [ ]:
train_bureau_desc = train_bureau_num.describe().toPandas().set_index("summary").T

In [ ]:
train_bureau_desc

summary,count,mean,stddev,min,max
SK_ID_BUREAU,24179741,6035775.229942206,493778.66238877864,5001710,6842888
SK_ID_CURR,24179741,278001.8189019064,102884.73624934933,100001,456255
DAYS_CREDIT,24179741,-1592.5780535862646,758.0397617119994,-2922,0
CREDIT_DAY_OVERDUE,24179741,0.8697354533284704,38.454514398772865,0,2792
DAYS_CREDIT_ENDDATE,23002240,-9.531257738376784,4830.976235687948,-42060.0,31131.0
DAYS_ENDDATE_FACT,18551098,-1273.3275881028712,712.6843961681687,-42023.0,0.0
AMT_CREDIT_MAX_OVERDUE,6633838,6119.993925508283,385190.6037847946,0.0,1.15987185E8
CNT_CREDIT_PROLONG,24179741,0.006598085562620377,0.09975485433550886,0,9
AMT_CREDIT_SUM,24179736,339459.1957923061,1483039.001709823,0.0,5.85E8
AMT_CREDIT_SUM_DEBT,20090664,87653.97114779244,612121.8014003853,-2014753.455,1.701E8


In [ ]:
train_bureau_desc = pd.merge(summary_df, train_bureau_desc, left_index=True, right_index=True).iloc[:,[3, 4, 5, 6, 0, 1, 2, 7]]
train_bureau_desc

,count,mean,stddev,min,25%,50%,75%,max
SK_ID_BUREAU,24179741,6035775.229942206,493778.66238877864,5001710,5731156.0,6070820.0,6424177.000,6842888
SK_ID_CURR,24179741,278001.8189019064,102884.73624934933,100001,186232.0,277654.0,366698.000,456255
DAYS_CREDIT,24179741,-1592.5780535862646,758.0397617119994,-2922,-2282.0,-1544.0,-978.000,0
CREDIT_DAY_OVERDUE,24179741,0.8697354533284704,38.454514398772865,0,0.0,0.0,0.000,2792
DAYS_CREDIT_ENDDATE,23002240,-9.531257738376784,4830.976235687948,-42060.0,-1674.0,-925.0,-71.000,31131.0
DAYS_ENDDATE_FACT,18551098,-1273.3275881028712,712.6843961681687,-42023.0,-1813.0,-1196.0,-708.000,0.0
AMT_CREDIT_MAX_OVERDUE,6633838,6119.993925508283,385190.6037847946,0.0,0.0,0.0,0.450,1.15987185E8
CNT_CREDIT_PROLONG,24179741,0.006598085562620377,0.09975485433550886,0,0.0,0.0,0.000,9
AMT_CREDIT_SUM,24179736,339459.1957923061,1483039.001709823,0.0,48757.5,114750.0,276043.500,5.85E8
AMT_CREDIT_SUM_DEBT,20090664,87653.97114779244,612121.8014003853,-2014753.455,0.0,0.0,0.000,1.701E8


- Outliers of Numerical Features

- Correlation

## Handle Missing Values and Outliers

- Handle missing value

In [ ]:
cols_to_fill = [
    'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE',
    'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_ANNUITY'
]


median_values = {item['column']: item['50%'] for item in summary_data}

for c in cols_to_fill:
    if c in median_values:
        median_value = median_values[c]
        train_bureau = train_bureau.withColumn(
            c,
            F.when(F.col(c).isNull(), median_value).otherwise(F.col(c))
        )

In [ ]:
check_missing_values(train_bureau)

There are no missing values in the dataset.


- Handle Outlier

In [ ]:
summary_data = calculate_quantile(train_bureau)

In [ ]:
summary_data

[{'column': 'SK_ID_BUREAU',
  '25%': 5731156.0,
  '50%': 6070820.0,
  '75%': 6424177.0},
 {'column': 'SK_ID_CURR', '25%': 186232.0, '50%': 277654.0, '75%': 366698.0},
 {'column': 'DAYS_CREDIT', '25%': -2282.0, '50%': -1544.0, '75%': -978.0},
 {'column': 'CREDIT_DAY_OVERDUE', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'DAYS_CREDIT_ENDDATE',
  '25%': -1674.0,
  '50%': -925.0,
  '75%': -71.0},
 {'column': 'DAYS_ENDDATE_FACT',
  '25%': -1813.0,
  '50%': -1196.0,
  '75%': -708.0},
 {'column': 'AMT_CREDIT_MAX_OVERDUE', '25%': 0.0, '50%': 0.0, '75%': 0.45},
 {'column': 'CNT_CREDIT_PROLONG', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'AMT_CREDIT_SUM',
  '25%': 48757.5,
  '50%': 114750.0,
  '75%': 276043.5},
 {'column': 'AMT_CREDIT_SUM_DEBT', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'AMT_CREDIT_SUM_LIMIT', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'AMT_CREDIT_SUM_OVERDUE', '25%': 0.0, '50%': 0.0, '75%': 0.0},
 {'column': 'DAYS_CREDIT_UPDATE',
  '25%': -1216.0,
  '50%'

In [ ]:
train_bureau_num = get_numeric(train_bureau)

In [ ]:
printInfo(train_bureau_num)

Entries count: 24179741
Data columns (total 15 columns)
root
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- CREDIT_DAY_OVERDUE: integer (nullable = true)
 |-- DAYS_CREDIT_ENDDATE: double (nullable = true)
 |-- DAYS_ENDDATE_FACT: double (nullable = true)
 |-- AMT_CREDIT_MAX_OVERDUE: double (nullable = true)
 |-- CNT_CREDIT_PROLONG: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)
 |-- AMT_CREDIT_SUM_LIMIT: double (nullable = true)
 |-- AMT_CREDIT_SUM_OVERDUE: double (nullable = true)
 |-- DAYS_CREDIT_UPDATE: integer (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- MONTHS_BALANCE: integer (nullable = true)



In [ ]:
for elm in summary_data:
  Q1, Q3 = elm["25%"], elm["75%"]
  iqr = Q3-Q1
  train_bureau_num = train_bureau_num.filter((col(elm["column"]) >= Q1 - 1.5 * iqr) & (col(elm["column"]) <= Q3 + 1.5 * iqr))

In [ ]:
printInfo(train_bureau_num)

Entries count: 15762761
Data columns (total 15 columns)
root
 |-- SK_ID_BUREAU: integer (nullable = true)
 |-- SK_ID_CURR: integer (nullable = true)
 |-- DAYS_CREDIT: integer (nullable = true)
 |-- CREDIT_DAY_OVERDUE: integer (nullable = true)
 |-- DAYS_CREDIT_ENDDATE: double (nullable = true)
 |-- DAYS_ENDDATE_FACT: double (nullable = true)
 |-- AMT_CREDIT_MAX_OVERDUE: double (nullable = true)
 |-- CNT_CREDIT_PROLONG: integer (nullable = true)
 |-- AMT_CREDIT_SUM: double (nullable = true)
 |-- AMT_CREDIT_SUM_DEBT: double (nullable = true)
 |-- AMT_CREDIT_SUM_LIMIT: double (nullable = true)
 |-- AMT_CREDIT_SUM_OVERDUE: double (nullable = true)
 |-- DAYS_CREDIT_UPDATE: integer (nullable = true)
 |-- AMT_ANNUITY: double (nullable = true)
 |-- MONTHS_BALANCE: integer (nullable = true)



In [ ]:
train_bureau = train_bureau.withColumn("MONTHS_BALANCE", -col("MONTHS_BALANCE"))

In [ ]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col

cols_to_transform = [
    'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE',
    'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG',
    'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT',
    'AMT_CREDIT_SUM_OVERDUE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY'
]

assembler = VectorAssembler(inputCols=cols_to_transform, outputCol="features_vector")
train_bureau = assembler.transform(train_bureau)

scaler = StandardScaler(inputCol="features_vector", outputCol="scaled_features", withMean=True, withStd=True)
scaler_model = scaler.fit(train_bureau)
train_bureau = scaler_model.transform(train_bureau)

train_bureau = train_bureau.withColumn("scaled_array", vector_to_array("scaled_features"))

for i, col_name in enumerate(cols_to_transform):
    train_bureau = train_bureau.withColumn(col_name, col("scaled_array")[i])

train_bureau = train_bureau.drop("features_vector", "scaled_features", "scaled_array")

train_bureau.show()

+------------+----------+-------------+---------------+-------------------+--------------------+-------------------+-------------------+----------------------+--------------------+--------------------+-------------------+--------------------+----------------------+---------------+------------------+--------------------+--------------+------+
|SK_ID_BUREAU|SK_ID_CURR|CREDIT_ACTIVE|CREDIT_CURRENCY|        DAYS_CREDIT|  CREDIT_DAY_OVERDUE|DAYS_CREDIT_ENDDATE|  DAYS_ENDDATE_FACT|AMT_CREDIT_MAX_OVERDUE|  CNT_CREDIT_PROLONG|      AMT_CREDIT_SUM|AMT_CREDIT_SUM_DEBT|AMT_CREDIT_SUM_LIMIT|AMT_CREDIT_SUM_OVERDUE|    CREDIT_TYPE|DAYS_CREDIT_UPDATE|         AMT_ANNUITY|MONTHS_BALANCE|STATUS|
+------------+----------+-------------+---------------+-------------------+--------------------+-------------------+-------------------+----------------------+--------------------+--------------------+-------------------+--------------------+----------------------+---------------+------------------+------------

### Feature Encoding

In [ ]:
train_bureau_cat = get_categorical(train_bureau)
train_bureau_cat.show()

+-------------+---------------+---------------+------+
|CREDIT_ACTIVE|CREDIT_CURRENCY|    CREDIT_TYPE|STATUS|
+-------------+---------------+---------------+------+
|       Active|     currency 1|    Credit card|     X|
|       Active|     currency 1|    Credit card|     0|
|       Active|     currency 1|    Credit card|     0|
|       Active|     currency 1|    Credit card|     0|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     C|
|       Closed|     currency 1|Consumer credit|     0|
|       Closed|     currency 1|Consumer credit|     0|
|       Cl

In [ ]:
nunique_dict = {col_name: train_bureau_cat.select(col_name).distinct().count() for col_name in train_bureau_cat.columns}
nunique_dict

{'CREDIT_ACTIVE': 4, 'CREDIT_CURRENCY': 4, 'CREDIT_TYPE': 14, 'STATUS': 8}

In [ ]:
columns = ['CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'CREDIT_TYPE', 'STATUS']

for col_name in columns:
    print(f"Unique values in column {col_name}:")
    train_bureau_cat.select(col_name).distinct().show()
    print("\n")

Unique values in column CREDIT_ACTIVE:
+-------------+
|CREDIT_ACTIVE|
+-------------+
|     Bad debt|
|         Sold|
|       Active|
|       Closed|
+-------------+



Unique values in column CREDIT_CURRENCY:
+---------------+
|CREDIT_CURRENCY|
+---------------+
|     currency 2|
|     currency 1|
|     currency 4|
|     currency 3|
+---------------+



Unique values in column CREDIT_TYPE:
+--------------------+
|         CREDIT_TYPE|
+--------------------+
|Loan for the purc...|
|Cash loan (non-ea...|
|           Microloan|
|     Consumer credit|
|Mobile operator loan|
|Another type of loan|
|            Mortgage|
|Loan for working ...|
|            Car loan|
|    Real estate loan|
|Unknown type of loan|
|Loan for business...|
|         Credit card|
|Loan for purchase...|
+--------------------+



Unique values in column STATUS:
+------+
|STATUS|
+------+
|     3|
|     0|
|     5|
|     C|
|     X|
|     1|
|     4|
|     2|
+------+





#### STATUS

In [ ]:
from pyspark.sql.types import IntegerType

train_bureau = train_bureau.withColumn(
    "STATUS",
    when(col("STATUS").isin('C', 'X', '0'), 0)
    .otherwise(col("STATUS").cast("int"))
    .cast(IntegerType())
)

#### CREDIT CURRENCY

In [ ]:
train_bureau = train_bureau.withColumn(
    'CREDIT_CURRENCY',
    when(col('CREDIT_CURRENCY') == 'currency 1', 1).otherwise(0)
)

### Class Imbalances

In [ ]:
from pyspark.sql.functions import count

# Get numeric columns using your function
numeric_cols = get_numeric(train_bureau)
train_bureau_num = train_bureau.select(*numeric_cols)

# Total rows for percentage calculation
total_rows = train_bureau.count()

# Loop through each numeric column and calculate distribution
for column in numeric_cols:
    print(f"Distribution for column {column}:")
    dist_df = train_bureau_num.groupBy(column).agg(
        (count("*") / total_rows * 100).alias("percentage")
    ).orderBy("percentage", ascending=False)
    dist_df.show(truncate=False)
    print("\n")

Distribution for column Column<'SK_ID_BUREAU'>:
+------------+---------------------+
|SK_ID_BUREAU|percentage           |
+------------+---------------------+
|5562162     |4.0116227878536826E-4|
|5045788     |4.0116227878536826E-4|
|6288336     |4.0116227878536826E-4|
|6801845     |4.0116227878536826E-4|
|5628326     |4.0116227878536826E-4|
|5136906     |4.0116227878536826E-4|
|5764482     |4.0116227878536826E-4|
|5775279     |4.0116227878536826E-4|
|5795381     |4.0116227878536826E-4|
|5819549     |4.0116227878536826E-4|
|5814293     |4.0116227878536826E-4|
|5819652     |4.0116227878536826E-4|
|5816390     |4.0116227878536826E-4|
|5951838     |4.0116227878536826E-4|
|5843463     |4.0116227878536826E-4|
|5968014     |4.0116227878536826E-4|
|5897757     |4.0116227878536826E-4|
|5983491     |4.0116227878536826E-4|
|5901079     |4.0116227878536826E-4|
|5996667     |4.0116227878536826E-4|
+------------+---------------------+
only showing top 20 rows



Distribution for column Column<'SK_I

#### Handling Imbalances

In [46]:
# Oversample minority class by duplicating rows (approximates SMOTE behavior by balancing without losing data)
from pyspark.sql.functions import col

# Convert target to binary
train_bureau = train_bureau.withColumn("CREDIT_DAY_OVERDUE_BINARY", (col("CREDIT_DAY_OVERDUE") > 0).cast("int"))

# Split data
majority = train_bureau.filter(col("CREDIT_DAY_OVERDUE_BINARY") == 0)
minority = train_bureau.filter(col("CREDIT_DAY_OVERDUE_BINARY") == 1)

# Calculate ratio
majority_count = majority.count()
minority_count = minority.count()
ratio = majority_count // minority_count
remainder = majority_count % minority_count

# Oversample minority class
oversampled_minority = minority
for _ in range(ratio - 1):
    oversampled_minority = oversampled_minority.union(minority)

if remainder > 0:
    oversampled_minority = oversampled_minority.union(minority.sample(withReplacement=True, fraction=remainder / minority_count, seed=42))

# Combine both
balanced_df = majority.union(oversampled_minority)

# Show class distribution
balanced_df.groupBy("CREDIT_DAY_OVERDUE_BINARY").count().show()

Py4JJavaError: An error occurred while calling o2289.showString.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 2 in stage 558.0 failed 1 times, most recent failure: Lost task 2.0 in stage 558.0 (TID 644) (192d38fa224c executor driver): java.lang.OutOfMemoryError: Java heap space

Driver stacktrace:
	at org.apache.spark.scheduler.DAGScheduler.failJobAndIndependentStages(DAGScheduler.scala:2856)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:2792)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:2791)
	at scala.collection.mutable.ResizableArray.foreach(ResizableArray.scala:62)
	at scala.collection.mutable.ResizableArray.foreach$(ResizableArray.scala:55)
	at scala.collection.mutable.ArrayBuffer.foreach(ArrayBuffer.scala:49)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:2791)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1(DAGS

In [ ]:
train_bureau.groupBy("CREDIT_DAY_OVERDUE").count().orderBy("count", ascending=False).show()

### Feature Selection

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml import Pipeline
from pyspark.sql.functions import col

# Step 1: Prepare data
# Convert target to float
train_bureau = train_bureau.withColumn("CREDIT_DAY_OVERDUE", col("CREDIT_DAY_OVERDUE").cast("float"))

# Drop unused column
features_cols = [col for col in train_bureau.columns if col not in ["CREDIT_DAY_OVERDUE", "CREDIT_TYPE"]]

# Step 2: Assemble features into a single vector
assembler = VectorAssembler(inputCols=features_cols, outputCol="features")

# Step 3: Initialize RandomForestRegressor
rf = RandomForestRegressor(featuresCol="features", labelCol="CREDIT_DAY_OVERDUE", numTrees=100, seed=42)

# Step 4: Create pipeline and fit model
pipeline = Pipeline(stages=[assembler, rf])
model = pipeline.fit(train_bureau)

# Step 5: Get feature importances
importances = model.stages[-1].featureImportances
feature_importance_list = list(zip(features_cols, importances))

# Step 6: Sort and display top features
sorted_features = sorted(feature_importance_list, key=lambda x: x[1], reverse=True)

print("Top features by importance:")
for name, score in sorted_features:
    print(f"{name}: {score:.4f}")